# Agent eval — baseline (`/chat`) vs агент (`/agent`)

Сравниваем чистый RAG и агента на одном golden-наборе из 18 вопросов
метриками RAGAS `faithfulness` и `answer_relevancy`.

| Cell | Что делает | Куда пишет |
|---|---|---|
| 1 | окружение + golden-набор | — |
| 2 | baseline: `POST /chat` | `notebooks/baseline_metrics.json` |
| 4 | агент: `POST /agent` + сравнение | `notebooks/agent_metrics.json` |

Перед запуском сервис должен быть поднят: `docker compose up -d`.
Прогон небыстрый: каждый вопрос — это вызов LLM, плюс LLM-judge на каждую метрику.


In [1]:
"""Cell 1 — окружение и golden-набор вопросов.

Ноутбук не должен зависеть от того, из какого каталога запущен kernel.
Jupyter по умолчанию делает cwd = каталог ноутбука (то есть notebooks/),
а от корня проекта зависят сразу три вещи: `.env` (его читает
pydantic-settings в app/core/config.py), импорт `app.*` и путь `notebooks/*.json`
при записи метрик. Поэтому корень ищем вверх по дереву, а не угадываем.
"""
import json
import os
import sys
from pathlib import Path

import httpx
from datasets import Dataset
from langchain_huggingface import HuggingFaceEmbeddings
from ragas import evaluate
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import Faithfulness, ResponseRelevancy


def find_repo_root(start: Path) -> Path:
    """Корень проекта — первый каталог вверх по дереву, где есть app/main.py."""
    for candidate in (start, *start.parents):
        if (candidate / "app" / "main.py").is_file():
            return candidate
    raise RuntimeError(
        f"Не нашёл корень проекта (app/main.py) выше {start}. "
        "Открой ноутбук из каталога task-api или запусти kernel оттуда."
    )


REPO_ROOT = find_repo_root(Path.cwd())
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print("рабочий каталог:", Path.cwd())

from app.llm import get_llm  # noqa: E402  (после правки sys.path)

BASE_URL = "http://localhost:8000"

"""Ячейка 2 — golden-датасет из 18 эталонных вопросов.

Структура: 13 английских и 4 русских вопроса по всем 10 разделам корпуса
(linear_model / tree / model_evaluation + ensemble / cross_validation /
preprocessing / compose / grid_search / impute / feature_selection) и
1 meta-вопрос про `about.md`. Поле `ground_truth_url_keywords` нужно для
Recall@k в Ячейке 4: считаем hit, если URL retrieved-чанка содержит
ожидаемое ключевое слово.
"""
GOLDEN = [
    # --- 7 EN in-corpus вопросов ---
    {
        "question": "How does Ridge regression handle multicollinearity?",
        "ground_truth": "Ridge adds an L2 penalty alpha * sum(w_i^2) to the loss, "
                        "which shrinks correlated coefficients toward each other.",
        "ground_truth_url_keywords": ["linear_model"],
    },
    {
        "question": "What does the alpha parameter control in Ridge?",
        "ground_truth": "Alpha controls regularization strength; larger alpha means "
                        "stronger penalty and smaller coefficients.",
        "ground_truth_url_keywords": ["linear_model"],
    },
    {
        "question": "What is the difference between Lasso and Ridge?",
        "ground_truth": "Lasso uses L1 penalty which can zero out coefficients (feature "
                        "selection); Ridge uses L2 which shrinks but never zeroes.",
        "ground_truth_url_keywords": ["linear_model"],
    },
    {
        "question": "What does min_samples_leaf control in a decision tree?",
        "ground_truth": "min_samples_leaf is the minimum number of samples required to be "
                        "at a leaf node; higher values prevent overfitting by limiting depth.",
        "ground_truth_url_keywords": ["tree"],
    },
    {
        "question": "When does a decision tree overfit?",
        "ground_truth": "Trees overfit when grown too deep without min_samples_leaf or "
                        "min_samples_split constraints, memorising training noise.",
        "ground_truth_url_keywords": ["tree"],
    },
    {
        "question": "What is the formula for precision?",
        "ground_truth": "precision = TP / (TP + FP). Fraction of positive predictions that "
                        "are actually positive.",
        "ground_truth_url_keywords": ["model_evaluation"],
    },
    {
        "question": "When is recall more important than precision?",
        "ground_truth": "Recall matters most when missing positives is costly: cancer "
                        "screening, fraud detection, anything where false negatives are "
                        "more harmful than false positives.",
        "ground_truth_url_keywords": ["model_evaluation"],
    },
    # --- 2 RU вопроса (тест мультиязычного retrieval) ---
    {
        "question": "Что такое L2-регуляризация?",
        "ground_truth": "L2-регуляризация добавляет к функции потерь штраф, "
                        "пропорциональный сумме квадратов коэффициентов модели. "
                        "Используется в Ridge.",
        "ground_truth_url_keywords": ["linear_model"],
    },
    {
        "question": "Что такое F1-мера?",
        "ground_truth": "F1 — гармоническое среднее precision и recall, "
                        "F1 = 2 * precision * recall / (precision + recall).",
        "ground_truth_url_keywords": ["model_evaluation"],
    },
    # --- 8 вопросов по расширенному корпусу ---
    {
        "question": "How does Random Forest reduce variance compared to a single decision tree?",
        "ground_truth": "Random Forest averages many decorrelated trees trained on bootstrap "
                        "samples, so individual tree variance cancels out.",
        "ground_truth_url_keywords": ["ensemble"],
    },
    {
        "question": "What is the difference between Gradient Boosting and Random Forest?",
        "ground_truth": "Gradient Boosting fits trees sequentially, each correcting the previous "
                        "ensemble's errors; Random Forest trains trees independently and averages them.",
        "ground_truth_url_keywords": ["ensemble"],
    },
    {
        "question": "Когда нужен StratifiedKFold вместо обычного KFold?",
        "ground_truth": "StratifiedKFold сохраняет доли классов в каждом фолде — это критично "
                        "для несбалансированных классов и маленьких выборок.",
        "ground_truth_url_keywords": ["cross_validation"],
    },
    {
        "question": "Why is feature scaling important for regularised linear models?",
        "ground_truth": "Регуляризация штрафует величину коэффициентов, поэтому неотскейленные "
                        "признаки штрафуются неравномерно; scaling делает alpha осмысленной.",
        "ground_truth_url_keywords": ["preprocessing"],
    },
    {
        "question": "How does a Pipeline help avoid data leakage?",
        "ground_truth": "Pipeline вызывает fit только на train-части и применяет уже обученные "
                        "трансформеры к тесту, поэтому статистики вроде среднего скалера не видят тест.",
        "ground_truth_url_keywords": ["compose"],
    },
    {
        "question": "Чем RandomizedSearchCV отличается от GridSearchCV?",
        "ground_truth": "GridSearchCV перебирает все комбинации параметров, RandomizedSearchCV — "
                        "заданное число случайных комбинаций; на больших пространствах второй быстрее.",
        "ground_truth_url_keywords": ["grid_search"],
    },
    {
        "question": "How does SimpleImputer fill missing values?",
        "ground_truth": "SimpleImputer заменяет пропуски статистикой, посчитанной на train "
                        "(mean, median, most_frequent или константа), и умеет добавлять индикаторы пропусков.",
        "ground_truth_url_keywords": ["impute"],
    },
    {
        "question": "What does SelectFromModel do?",
        "ground_truth": "SelectFromModel оставляет признаки, чья важность или модуль коэффициента "
                        "выше порога, вычисленного по обученному estimator.",
        "ground_truth_url_keywords": ["feature_selection"],
    },
    # --- 1 meta-вопрос (тест about.md) ---
    {
        "question": "Что ты умеешь?",
        "ground_truth": "Отвечаю на вопросы по трём разделам scikit-learn: линейные "
                        "модели, деревья решений, метрики качества.",
        "ground_truth_url_keywords": ["about.md", "local"],
    },
]

# Ячейки ниже обращаются к ALL — алиас, чтобы их код не пришлось править
ALL = GOLDEN
print(f"вопросов в наборе: {len(ALL)}")


C:\Users\DW\anaconda3\envs\task-api\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


рабочий каталог: C:\Users\DW\Desktop\mentoring\task-api
вопросов в наборе: 18


## Cell 2 — baseline: ходим в `/chat` и меряем качество чистого RAG

На multi-hop вопросах baseline проваливается — это ожидаемо. Цифры идут
в `baseline_metrics.json` для последующего сравнения с agent в Cell 4.


In [2]:
from datasets import Dataset
from langchain_huggingface import HuggingFaceEmbeddings
from ragas import evaluate
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import Faithfulness, ResponseRelevancy

from app.llm import get_llm

baseline_results = []

# Дёргаем /chat синхронно на каждом вопросе и копим контекст для RAGAS
for i, item in enumerate(ALL):
    r = httpx.post(f"{BASE_URL}/chat", json={"question": item["question"]}, timeout=120)
    # без raise_for_status() ошибка сервера превращается в невнятный KeyError
    r.raise_for_status()
    data = r.json()
    baseline_results.append({
        "user_input": item["question"],
        "response": data["answer"],
        "retrieved_contexts": [s["full_context"] for s in data["sources"]],
        "reference": item.get("ground_truth", ""),
    })
    print(f"[{i + 1}/{len(ALL)}] baseline ok")

# Считаем RAGAS-метрики через LLM-judge — обёртки требуют совместимости
llm = LangchainLLMWrapper(get_llm())
emb = LangchainEmbeddingsWrapper(
    HuggingFaceEmbeddings(
        model_name="intfloat/multilingual-e5-small",
        encode_kwargs={"normalize_embeddings": True},
    )
)
baseline_scores = evaluate(
    dataset=Dataset.from_list(baseline_results),
    metrics=[Faithfulness(llm=llm), ResponseRelevancy(llm=llm, embeddings=emb)],
)
baseline_df = baseline_scores.to_pandas()

# write_text() НЕ создаёт родительские каталоги: если cwd не корень проекта,
# относительный путь превращается в notebooks/notebooks/... и падает с
# FileNotFoundError. Проверяем заранее и говорим, что делать.
metrics_path = Path("notebooks/baseline_metrics.json")
assert metrics_path.parent.is_dir(), (
    f"Нет каталога {metrics_path.parent.resolve()} — сначала выполни Cell 1, "
    "он переходит в корень проекта"
)
metrics_path.write_text(
    json.dumps({
        "faithfulness": float(baseline_df["faithfulness"].mean()),
        "answer_relevancy": float(baseline_df["answer_relevancy"].mean()),
        "n_questions": len(ALL),
    }, indent=2, ensure_ascii=False),
    encoding="utf-8",
)
print(baseline_df[["faithfulness", "answer_relevancy"]].mean())


[1/18] baseline ok


[2/18] baseline ok


[3/18] baseline ok


[4/18] baseline ok


[5/18] baseline ok


...

Evaluating:  92%|█████████▏| 33/36 [00:19<00:02,  1.14it/s]

Evaluating:  97%|█████████▋| 35/36 [00:20<00:00,  1.30it/s]

Evaluating: 100%|██████████| 36/36 [00:41<00:00,  5.68s/it]

Evaluating: 100%|██████████| 36/36 [00:41<00:00,  1.15s/it]

faithfulness        0.888889
answer_relevancy    0.947928
dtype: float64


## Cell 4 — агент: те же вопросы в `/agent` и сравнение с baseline

Агент отвечает дольше (несколько шагов LLM + вызовы инструментов),
поэтому таймаут на запрос больше. В конце печатается сравнение с
`baseline_metrics.json`.


In [3]:
"""Cell 4 — агент: те же вопросы в /agent + сравнение с baseline.

ВАЖНО про контекст. `/agent` кладёт в `full_context` текст, который вернул
инструмент `documentation_search`, а это уже сгенерированный ответ
(`answer` + `Sources:`), а не найденные чанки. Поэтому faithfulness у агента
структурно завышен и напрямую не сравним с baseline из Cell 2: цифры агента
читаем как верхнюю границу. Чтобы сравнение стало честным, /agent должен
отдавать реально найденные чанки — это отдельная правка бэкенда.
"""
agent_results = []
agent_extra = []

for i, item in enumerate(ALL):
    r = httpx.post(f"{BASE_URL}/agent", json={"question": item["question"]}, timeout=300)

    if r.status_code != 200:
        # 422 — вопрос отклонили guardrails, 503 — LLM-провайдер недоступен
        print(f"[{i + 1}/{len(ALL)}] /agent -> HTTP {r.status_code}: {r.text[:120]}")
        agent_results.append({
            "user_input": item["question"],
            "response": "",
            "retrieved_contexts": [],
            "reference": item.get("ground_truth", ""),
        })
        agent_extra.append({"iterations": 0, "tool_calls": 0, "status": r.status_code})
        continue

    data = r.json()
    agent_results.append({
        "user_input": item["question"],
        "response": data["answer"],
        "retrieved_contexts": [s["full_context"] for s in data["sources"]],
        "reference": item.get("ground_truth", ""),
    })
    agent_extra.append({
        "iterations": data.get("iterations", 0),
        "tool_calls": len(data.get("trace", [])),
        "status": 200,
    })
    print(f"[{i + 1}/{len(ALL)}] agent ok | итераций: {data.get('iterations', 0)} | "
          f"tool-вызовов: {len(data.get('trace', []))} | источников: {len(data['sources'])}")

# --- те же метрики, что и в Cell 2 ---
llm = LangchainLLMWrapper(get_llm())
emb = LangchainEmbeddingsWrapper(
    HuggingFaceEmbeddings(
        model_name="intfloat/multilingual-e5-small",
        encode_kwargs={"normalize_embeddings": True},
    )
)
agent_scores = evaluate(
    dataset=Dataset.from_list(agent_results),
    metrics=[Faithfulness(llm=llm), ResponseRelevancy(llm=llm, embeddings=emb)],
)
agent_df = agent_scores.to_pandas()

agent_metrics = {
    "faithfulness": float(agent_df["faithfulness"].mean()),
    "answer_relevancy": float(agent_df["answer_relevancy"].mean()),
    "n_questions": len(ALL),
    "avg_iterations": sum(e["iterations"] for e in agent_extra) / len(agent_extra),
    "avg_tool_calls": sum(e["tool_calls"] for e in agent_extra) / len(agent_extra),
    "failed_requests": sum(1 for e in agent_extra if e["status"] != 200),
}
metrics_path = Path("notebooks/agent_metrics.json")
assert metrics_path.parent.is_dir(), (
    f"Нет каталога {metrics_path.parent.resolve()} — сначала выполни Cell 1"
)
metrics_path.write_text(
    json.dumps(agent_metrics, indent=2, ensure_ascii=False),
    encoding="utf-8",
)


# --- сравнение с baseline ---
def _fmt(value):
    return f"{value:.3f}" if isinstance(value, (int, float)) else "n/a"


baseline_path = Path("notebooks/baseline_metrics.json")
print()
if baseline_path.exists():
    baseline_metrics = json.loads(baseline_path.read_text(encoding="utf-8"))
    print("=== baseline (/chat) vs agent (/agent) ===")
    for key in ("faithfulness", "answer_relevancy"):
        b, a = baseline_metrics.get(key), agent_metrics.get(key)
        line = f"  {key:18} baseline={_fmt(b)}  agent={_fmt(a)}"
        if isinstance(b, (int, float)) and isinstance(a, (int, float)):
            line += f"  Δ={a - b:+.3f}"
        print(line)
    print(f"  итераций агента (среднее): {agent_metrics['avg_iterations']:.1f}")
    print(f"  tool-вызовов (среднее):    {agent_metrics['avg_tool_calls']:.1f}")
    print(f"  неуспешных запросов:       {agent_metrics['failed_requests']}")
else:
    print("notebooks/baseline_metrics.json не найден — сначала прогони Cell 2")

print()
print(agent_df[["faithfulness", "answer_relevancy"]].mean())


[1/18] agent ok | итераций: 2 | tool-вызовов: 1 | источников: 4


[2/18] agent ok | итераций: 2 | tool-вызовов: 1 | источников: 4


[3/18] agent ok | итераций: 2 | tool-вызовов: 1 | источников: 4


[4/18] agent ok | итераций: 2 | tool-вызовов: 1 | источников: 4


[5/18] agent ok | итераций: 2 | tool-вызовов: 1 | источников: 4


...

Evaluating:  92%|█████████▏| 33/36 [00:18<00:01,  1.84it/s]

Evaluating:  97%|█████████▋| 35/36 [00:19<00:00,  2.26it/s]

Evaluating: 100%|██████████| 36/36 [00:20<00:00,  1.42it/s]

Evaluating: 100%|██████████| 36/36 [00:20<00:00,  1.74it/s]


=== baseline (/chat) vs agent (/agent) ===
  faithfulness       baseline=0.889  agent=0.847  Δ=-0.042
  answer_relevancy   baseline=0.948  agent=0.904  Δ=-0.043
  итераций агента (среднее): 1.9
  tool-вызовов (среднее):    0.9
  неуспешных запросов:       0

faithfulness        0.847222
answer_relevancy    0.904499
dtype: float64
